# VLM — Estimating Box Distance with a Depth Map

Generate a depth map from an image, then pass both the original image and depth map to Gemini to estimate how far away each box is.

In [ ]:
!pip install google-generativeai transformers torch pillow httpx matplotlib python-dotenv

In [ ]:
import os, io, json, httpx
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv
from transformers import pipeline
import google.generativeai as genai

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

vlm = genai.GenerativeModel("gemini-2.0-flash")
depth_pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")

In [ ]:
# --- Load image and generate depth map ---

IMAGE_URL = "https://images.unsplash.com/photo-1586528116311-ad8dd3c8310d?w=800"

image_data = httpx.get(IMAGE_URL).content
original = Image.open(io.BytesIO(image_data)).convert("RGB")

# Generate depth map (brighter = closer, darker = farther)
depth_result = depth_pipe(original)
depth_map = depth_result["depth"]  # PIL Image

In [ ]:
# --- Display original image and depth map side by side ---

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.imshow(original)
ax1.set_title("Original Image")
ax1.axis("off")

ax2.imshow(depth_map, cmap="inferno")
ax2.set_title("Depth Map (bright = close, dark = far)")
ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# --- Send BOTH images to Gemini for analysis ---

# Convert depth map to RGB so Gemini can read it
depth_rgb = depth_map.convert("RGB")

PROMPT = """
You are given two images:
1. The ORIGINAL photo of a scene with boxes/pallets/cartons.
2. A DEPTH MAP of the same scene where brighter areas are closer to the
   camera and darker areas are farther away.

Use BOTH images together to identify every box, carton, or pallet visible.
Use the depth map to improve your distance estimates.

For each box, provide:
- distance_ft: estimated distance from the camera in feet (use the depth map)
- size: small / medium / large
- position: where in the frame (left/center/right, foreground/midground/background)
- labels: any visible text or markings, or "none"
- description: one-line description

Respond ONLY with this JSON:
{
  "box_count": <int>,
  "boxes": [
    {
      "id": 1,
      "distance_ft": <float>,
      "size": "small | medium | large",
      "position": "<position>",
      "labels": "<text or none>",
      "description": "<brief>"
    }
  ],
  "scene_description": "<one sentence>"
}
"""

response = vlm.generate_content([original, depth_rgb, PROMPT])

# Parse JSON from response
text = response.text.strip()
if text.startswith("```"):
    text = text.split("\n", 1)[1]
if text.endswith("```"):
    text = text.rsplit("```", 1)[0]

result = json.loads(text)

In [ ]:
# --- Results ---

print(f"Scene: {result['scene_description']}")
print(f"Boxes found: {result['box_count']}\n")

for box in result["boxes"]:
    print(f"  Box #{box['id']}  —  ~{box['distance_ft']} ft away")
    print(f"    Size: {box['size']}  |  Position: {box['position']}")
    print(f"    Labels: {box['labels']}")
    print(f"    {box['description']}\n")

print("\n--- Full JSON ---")
print(json.dumps(result, indent=2))